# Encoder study on a GPU

Which encoder represents Persian well enough to build on — measured, not taken
from model cards.

On a free T4 the full 3600-image benchmark takes a few minutes per encoder.
The same run on a CPU takes hours, which is why the numbers for the paper
should come from here.

**Kaggle**: Settings → Accelerator → *GPU T4 x2*, and Internet **ON**.
**Colab**: Runtime → Change runtime type → *T4 GPU*.

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU. Turn on the accelerator and restart."
print(torch.cuda.get_device_name(0))

## 1. Install

`transformers` is pinned below 5 on purpose: jina-clip-v2's remote code imports
`clip_loss`, which 5.x removed. `einops` and `timm` are needed by that same
remote code.

In [ ]:
!pip install -q "transformers>=4.40,<5" einops timm

## 2. Get the code and the benchmark

Captions come from the project page, images from a single tarball — about
330 MB together. Neither is stored in the repository.

In [ ]:
!git clone -q https://github.com/zhrzkrneng/Persian-hate-speech-detection.git
%cd Persian-hate-speech-detection
!python -m scripts.download_xm3600

## 3. Run every encoder, both languages

English is the **control**. An encoder that scores well on English and badly on
Persian has a language problem; one that scores badly on both has a different
problem, and the two must not be confused.

Every encoder sees the same images, so the comparison is fair.

`|| true` keeps the loop going if one model fails to load — a gated or moved
model should not cost the runs that would have succeeded.

In [ ]:
%%bash
for encoder in clip-l14 jina-v2 clipfa; do
  for language in fa en; do
    echo "=== $encoder / $language ==="
    python -m scripts.eval_retrieval \
      --encoder "$encoder" --language "$language" \
      --device cuda --batch-size 128 || echo "FAILED: $encoder/$language"
  done
done

## 4. The table

In [ ]:
!python -m scripts.summarise_retrieval

## How to read it

**Chance R@1 is 100/N percent.** Over all 3600 images that is 0.03%, so a score
of 0.1% is not a small result — it is noise. The summary prints chance beside
every row for exactly this reason.

Two things that belong in the paper if jina-clip-v2 wins:

- it loads code from its model repository (`trust_remote_code`)
- it is licensed **CC BY-NC 4.0** — fine for research, not for a product

## Keeping the results

Each JSON records the encoder, the model id, the language, and how many images
and captions it scored, so any number can be traced back to the run that
produced it.

In [ ]:
import json, pathlib
for path in sorted(pathlib.Path("runs/retrieval").glob("*.json")):
    print(path, json.loads(path.read_text())["results"])